# 📊 Pipeline Performance Comparison
**Dataset:** mudah.my Malaysia Property Listings

| Pipeline | Method |
|----------|--------|
| 1 | Pandas Baseline (unoptimized) |
| 2 | Pandas Optimized |
| 3 | Polars |
| 4 | DuckDB |

**Metrics:** processing time (s), CPU (%), memory (MB), throughput (records/s)

## Step 1 — Install & Import

In [1]:
!pip install -q polars duckdb psutil

import pandas as pd
import polars as pl
import duckdb
import psutil, os, time, tracemalloc, gc
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

proc = psutil.Process(os.getpid())
proc.cpu_percent(interval=None)   # warm up

print('✅ All packages ready.')

✅ All packages ready.


## Step 2 — Upload CSV

In [2]:
CSV_FILE = 'cleaned_data.csv'

with open(CSV_FILE, 'r', encoding='utf-8') as f:
    total_records = sum(1 for line in f) - 1

## Step 3 — Helper: Performance Metrics Collection

In [3]:
all_run_records = []   # stores every individual run
all_metrics     = []   # stores averaged metrics per method


def run_benchmark(method_name, fn, n_rows_fn=None, runs=3):

    print(f'\n  Running: {method_name} ({runs} runs)...')

    all_times, all_cpu, all_mem, all_tput = [], [], [], []
    result = None

    for run in range(1, runs + 1):
        gc.collect()

        tracemalloc.start()
        proc.cpu_percent(interval=None)
        t_start = time.perf_counter()

        result = fn()

        elapsed   = time.perf_counter() - t_start
        cpu_pct   = proc.cpu_percent(interval=None)
        _, peak   = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memory_mb = peak / 1024**2

        n_rows    = n_rows_fn(result) if n_rows_fn else total_records
        throughput = n_rows / elapsed if elapsed > 0 else 0

        all_times.append(elapsed)
        all_cpu.append(cpu_pct)
        all_mem.append(memory_mb)
        all_tput.append(throughput)

        all_run_records.append({
            'method':                     method_name,
            'run':                        run,
            'time_sec':                   round(elapsed, 4),
            'cpu_percent':                round(cpu_pct, 2),
            'memory_mb':                  round(memory_mb, 2),
            'throughput_records_per_sec': round(throughput, 0),
            'rows_processed':             n_rows,
        })

        print(f'    Run {run}/{runs} → {elapsed:.4f}s | '
              f'CPU: {cpu_pct:.1f}% | '
              f'Mem: {memory_mb:.2f}MB | '
              f'Throughput: {throughput:,.0f} rec/s | '
              f'Rows: {n_rows:,}')

        gc.collect()
        time.sleep(1)

    avg = {
        'method':                     method_name,
        'time_sec':                   round(sum(all_times) / runs, 4),
        'time_min':                   round(min(all_times), 4),
        'time_max':                   round(max(all_times), 4),
        'cpu_percent':                round(sum(all_cpu) / runs, 2),
        'memory_mb':                  round(sum(all_mem) / runs, 2),
        'throughput_records_per_sec': round(sum(all_tput) / runs, 0),
        'rows_processed':             n_rows_fn(result) if n_rows_fn else total_records,
    }

    print(f'    ── Avg: {avg["time_sec"]}s | Min: {avg["time_min"]}s | Max: {avg["time_max"]}s')
    return avg, result


print('✅ Benchmark helper ready.')

✅ Benchmark helper ready.


## Step 4 — Pandas Baseline (Unoptimized)

In [4]:
# warm up
_ = pd.read_csv(CSV_FILE).head(100)
proc.cpu_percent(interval=None)


def pandas_baseline():
    """
    UNOPTIMIZED — no usecols, no dtype, no categorical.
    Data already clean, no dropna/dedup needed.
    """
    df = pd.read_csv(CSV_FILE)

    df['price_rm'] = pd.to_numeric(df['price_rm'], errors='coerce')

    # Same task as all other methods
    mask_house = df['property_type'].astype(str).str.contains(
        'house|bungalow|terrace|semi.d|villa|townhouse', case=False, na=False
    )
    df_house = df.loc[mask_house].copy()

    agg = (
        df_house.groupby('state')['price_rm']
        .agg(['mean', 'median', 'count'])
        .sort_values('mean', ascending=False)
    )

    df['price_bucket'] = pd.cut(
        df['price_rm'],
        bins=[0, 200_000, 500_000, 1_000_000, float('inf')],
        labels=['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M'],
        right=False
    )

    return {'df': df, 'df_house': df_house, 'agg': agg}


metrics_baseline, result_baseline = run_benchmark(
    'pandas_baseline',
    pandas_baseline,
    n_rows_fn=lambda r: len(r['df']),
)
all_metrics.append(metrics_baseline)

# ── Print results ─────────────────────────────────────────────────────────────
df_b     = result_baseline['df']
house_b  = result_baseline['df_house']
agg_b    = result_baseline['agg']

print('=' * 50)
print('  PANDAS BASELINE RESULTS')
print('=' * 50)
print(f'  Total Processing Time : {metrics_baseline["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_baseline["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_baseline["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_baseline["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_b):,}')
print('=' * 50)

print(f'\nTotal Houses Filtered: {len(house_b):,} rows\n')

print('Price Summary by State for Houses:')
print(f"  {'State':<20}   {'Mean Price':<18}   {'Median Price':<18}   {'Count':<8}")
for state, row in agg_b.head(5).iterrows():
    print(f"  {str(state):<20}   RM {row['mean']:>13,.2f}   RM {row['median']:>13,.2f}   {int(row['count']):>8,}")

print('\nDistribution by Price Bucket:')
bucket_order = ['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M']
bucket_counts = df_b['price_bucket'].value_counts().sort_index()
for b in bucket_order:
    print(f'  🔹 {b:<15} : {bucket_counts.get(b, 0):>6,} properties')

# Save for performance_before.csv
pd.DataFrame([metrics_baseline]).to_csv('performance_before.csv', index=False)
print('\n✅ Saved: performance_before.csv')

/tmp/ipykernel_2522/3910698144.py:2: DtypeWarning: Columns (0,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  _ = pd.read_csv(CSV_FILE).head(100)



  Running: pandas_baseline (3 runs)...


/tmp/ipykernel_2522/3910698144.py:11: DtypeWarning: Columns (0,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE)


    Run 1/3 → 11.1974s | CPU: 47.0% | Mem: 251.16MB | Throughput: 8,970 rec/s | Rows: 100,442


/tmp/ipykernel_2522/3910698144.py:11: DtypeWarning: Columns (0,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE)


    Run 2/3 → 4.1934s | CPU: 91.7% | Mem: 251.16MB | Throughput: 23,952 rec/s | Rows: 100,442


/tmp/ipykernel_2522/3910698144.py:11: DtypeWarning: Columns (0,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE)


    Run 3/3 → 4.6205s | CPU: 97.4% | Mem: 251.17MB | Throughput: 21,738 rec/s | Rows: 100,442
    ── Avg: 6.6705s | Min: 4.1934s | Max: 11.1974s
  PANDAS BASELINE RESULTS
  Total Processing Time : 6.6705 seconds
  CPU Usage             : 78.7 %
  Memory Usage          : 251.17 MB
  Throughput            : 18,220 records/second
  Rows Processed        : 100,442

Total Houses Filtered: 54,653 rows

Price Summary by State for Houses:
  State                  Mean Price           Median Price         Count   
  Penang                 RM  2,017,240.99   RM    890,000.00      3,228
  Sabah                  RM  1,728,408.90   RM    900,000.00      2,761
  Putrajaya              RM  1,482,884.64   RM  1,290,000.00        685
  Kuala Lumpur           RM  1,392,537.10   RM    780,000.00      3,123
  Selangor               RM  1,106,589.57   RM    604,000.00      6,673

Distribution by Price Bucket:
  🔹 < RM 200k       : 15,909 properties
  🔹 RM 200k-500k    : 43,468 properties
  🔹 RM 500k-1M    

## Step 5 — Pandas Optimized

In [5]:
# warm up
_ = pd.read_csv(CSV_FILE).head(100)
proc.cpu_percent(interval=None)


def pandas_optimized():
    """
    OPTIMIZED pandas pipeline.
    1. usecols        — load only 13 needed columns instead of all 16
    2. Explicit dtype — skip type inference on read_csv
    3. pd.Categorical — low-cardinality strings → integer codes (less RAM, faster groupby)
    4. .str.strip().str.title() — vectorised C-speed string cleaning
    5. Vectorised filter — boolean mask instead of query string
    6. groupby observed=True — skip empty categories
    7. pd.cut — vectorised binning
    """
    NEEDED_COLS = [
        'listing_id', 'description', 'property_type', 'location', 'state',
        'price_rm', 'mortgage_est_rm', 'land_title',
        'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type'
    ]

    # peek at available columns
    existing_cols = [c for c in NEEDED_COLS
                     if c in pd.read_csv(CSV_FILE, nrows=0).columns]

    # Opt 1 + 2: usecols + dtype
    df = pd.read_csv(
        CSV_FILE,
        usecols=existing_cols,
        dtype={'listing_id': 'str'},
        engine='c',
        na_values=['', 'None', 'nan'],
    )

    # Convert numeric safely
    for col in ['price_rm', 'mortgage_est_rm', 'size_sqft', 'bedrooms', 'bathrooms']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Opt 3: pd.Categorical
    for col in ['property_type', 'state', 'tenure', 'seller_type', 'land_title']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Opt 4: vectorised string cleaning
    if 'location' in df.columns:
        df['location'] = df['location'].str.strip().str.title()

    # Opt 5: vectorised filter
    mask_house = df['property_type'].astype(str).str.contains(
        'house|bungalow|terrace|semi.d|villa|townhouse', case=False, na=False
    )
    df_house = df.loc[mask_house].copy()

    # Opt 6: groupby observed=True
    agg = (
        df_house.groupby('state', observed=True)['price_rm']
        .agg(['mean', 'median', 'count'])
        .sort_values('mean', ascending=False)
    )

    # Opt 7: pd.cut
    df['price_bucket'] = pd.cut(
        df['price_rm'],
        bins=[0, 200_000, 500_000, 1_000_000, float('inf')],
        labels=['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M'],
        right=False
    )

    return {'df': df, 'df_house': df_house, 'agg': agg}


metrics_optimized, result_optimized = run_benchmark(
    'pandas_optimized',
    pandas_optimized,
    n_rows_fn=lambda r: len(r['df']),
)
all_metrics.append(metrics_optimized)

# ── Print results ─────────────────────────────────────────────────────────────
df_o    = result_optimized['df']
house_o = result_optimized['df_house']
agg_o   = result_optimized['agg']

print('=' * 50)
print('  PANDAS OPTIMIZATION RESULTS')
print('=' * 50)
print(f'  Total Processing Time : {metrics_optimized["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_optimized["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_optimized["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_optimized["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_o):,}')
print('=' * 50)

print(f'\nTotal Houses Filtered: {len(house_o):,} rows\n')

print('Price Summary by State for Houses:')
print(f"  {'State':<15}   {'Mean Price':<15}   {'Median Price':<15}   {'Count':<8}")
for state, row in agg_o.head(5).iterrows():
    mean_str = f"RM {row['mean']:,.2f}"
    med_str  = f"RM {row['median']:,.2f}"
    print(f"  {str(state):<15}   {mean_str:<15}   {med_str:<15}   {int(row['count']):,}")

print('\nDistribution by Price Bucket:')
bucket_order  = ['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M']
bucket_counts = df_o['price_bucket'].value_counts().sort_index()
for b in bucket_order:
    print(f'  🔹 {b:<15} : {bucket_counts.get(b, 0):>6,} properties')

/tmp/ipykernel_2522/1856629342.py:2: DtypeWarning: Columns (0,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  _ = pd.read_csv(CSV_FILE).head(100)



  Running: pandas_optimized (3 runs)...


/tmp/ipykernel_2522/1856629342.py:28: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 1/3 → 3.7298s | CPU: 100.0% | Mem: 213.61MB | Throughput: 26,930 rec/s | Rows: 100,442


/tmp/ipykernel_2522/1856629342.py:28: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 2/3 → 5.0461s | CPU: 98.4% | Mem: 213.61MB | Throughput: 19,905 rec/s | Rows: 100,442


/tmp/ipykernel_2522/1856629342.py:28: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 3/3 → 3.9322s | CPU: 100.0% | Mem: 213.61MB | Throughput: 25,544 rec/s | Rows: 100,442
    ── Avg: 4.236s | Min: 3.7298s | Max: 5.0461s
  PANDAS OPTIMIZATION RESULTS
  Total Processing Time : 4.2360 seconds
  CPU Usage             : 99.5 %
  Memory Usage          : 213.61 MB
  Throughput            : 24,126 records/second
  Rows Processed        : 100,442

Total Houses Filtered: 54,653 rows

Price Summary by State for Houses:
  State             Mean Price        Median Price      Count   
  Penang            RM 2,017,240.99   RM 890,000.00     3,228
  Sabah             RM 1,728,408.90   RM 900,000.00     2,761
  Putrajaya         RM 1,482,884.64   RM 1,290,000.00   685
  Kuala Lumpur      RM 1,392,537.10   RM 780,000.00     3,123
  Selangor          RM 1,106,589.57   RM 604,000.00     6,673

Distribution by Price Bucket:
  🔹 < RM 200k       : 15,909 properties
  🔹 RM 200k-500k    : 43,468 properties
  🔹 RM 500k-1M      : 25,193 properties
  🔹 > RM 1M         : 15,872 propertie

## Step 6 — Polars

In [6]:
proc.cpu_percent(interval=None)   # warm up


def polars_pipeline():
    lf = pl.scan_csv(
        CSV_FILE,
        infer_schema_length=10000,
        null_values=['', 'nan', 'None', 'N/A', 'Unknown'],
        schema_overrides={
            'listing_id':      pl.Utf8,
            'price_rm':        pl.Utf8,
            'bedrooms':        pl.Utf8,
            'bathrooms':       pl.Utf8,
            'size_sqft':       pl.Utf8,
            'mortgage_est_rm': pl.Utf8,
        },
        ignore_errors=True,
    )

    lf = lf.with_columns([
        pl.col('price_rm' ).cast(pl.Float64, strict=False),
        pl.col('bedrooms' ).cast(pl.Float64, strict=False),
        pl.col('bathrooms').cast(pl.Float64, strict=False),
        pl.col('size_sqft').cast(pl.Float64, strict=False),
    ])

    lf_houses = lf.filter(
        pl.col('property_type').str.to_lowercase().str.contains(
            'house|bungalow|terrace|semi.d|villa|townhouse'
        )
    )

    state_summary = (
        lf_houses
        .group_by('state')
        .agg([
            pl.mean('price_rm'   ).alias('mean_price'),
            pl.median('price_rm' ).alias('median_price'),
            pl.count('listing_id').alias('count'),
        ])
        .sort('mean_price', descending=True)
    )

    lf_buckets = lf_houses.with_columns(
        pl.when(pl.col('price_rm') < 200_000)
          .then(pl.lit('< RM 200k'))
          .when(pl.col('price_rm') < 500_000)
          .then(pl.lit('RM 200k-500k'))
          .when(pl.col('price_rm') < 1_000_000)
          .then(pl.lit('RM 500k-1M'))
          .otherwise(pl.lit('> RM 1M'))
          .alias('bucket')
    )
    bucket_dist = (
        lf_buckets
        .group_by('bucket')
        .agg(pl.count('listing_id').alias('count'))
        .sort('bucket')
    )

    df_houses  = lf_houses.collect()
    df_state   = state_summary.collect()
    df_buckets = bucket_dist.collect()

    return {'df_houses': df_houses, 'df_state': df_state, 'df_buckets': df_buckets}


metrics_polars, result_polars = run_benchmark(
    'polars',
    polars_pipeline,
    n_rows_fn=lambda r: len(r['df_houses']),
)
all_metrics.append(metrics_polars)

# ── Print results ─────────────────────────────────────────────────────────────
df_state_p   = result_polars['df_state']
df_houses_p  = result_polars['df_houses']
df_buckets_p = result_polars['df_buckets']

print('=' * 50)
print('  POLARS OPTIMIZATION RESULTS')
print('=' * 50)
print(f'  Total Processing Time : {metrics_polars["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_polars["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_polars["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_polars["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_houses_p):,}')
print('=' * 50)

print(f'\nTotal Houses Filtered: {len(df_houses_p):,} rows')

print('\nPrice Summary by State for Houses:')
print(f'{"State":<20} {"Mean Price":>18} {"Median Price":>18} {"Count":>8}')
for row in df_state_p.head(5).iter_rows(named=True):
    print(f'{row["state"]:<20} RM {row["mean_price"]:>15,.2f} RM {row["median_price"]:>15,.2f} {row["count"]:>8,}')

print('\nDistribution by Price Bucket:')
bucket_order = ['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M']
bucket_map   = {r['bucket']: r['count'] for r in df_buckets_p.iter_rows(named=True)}
for b in bucket_order:
    print(f'  ◆ {b:<15} : {bucket_map.get(b, 0):>6,} properties')


  Running: polars (3 runs)...
    Run 1/3 → 0.9320s | CPU: 113.7% | Mem: 0.09MB | Throughput: 58,641 rec/s | Rows: 54,653
    Run 2/3 → 0.8334s | CPU: 124.7% | Mem: 0.04MB | Throughput: 65,582 rec/s | Rows: 54,653
    Run 3/3 → 0.7450s | CPU: 132.7% | Mem: 0.04MB | Throughput: 73,360 rec/s | Rows: 54,653
    ── Avg: 0.8368s | Min: 0.745s | Max: 0.932s
  POLARS OPTIMIZATION RESULTS
  Total Processing Time : 0.8368 seconds
  CPU Usage             : 123.7 %
  Memory Usage          : 0.06 MB
  Throughput            : 65,861 records/second
  Rows Processed        : 54,653

Total Houses Filtered: 54,653 rows

Price Summary by State for Houses:
State                        Mean Price       Median Price    Count
Penang               RM    2,017,240.99 RM      890,000.00    3,228
Sabah                RM    1,728,408.90 RM      900,000.00    2,761
Putrajaya            RM    1,482,884.64 RM    1,290,000.00      685
Kuala Lumpur         RM    1,392,537.10 RM      780,000.00    3,123
Selangor     

## Step 7 — DuckDB

In [7]:
proc.cpu_percent(interval=None)   # warm up

# ── 1. Measure total records from CSV safely ──────────────────────────────────
with open(CSV_FILE, 'r', encoding='utf-8') as f:
    total_records = sum(1 for line in f) - 1


def duckdb_pipeline():
    con = duckdb.connect(database=':memory:')

    # House filter + price summary by state
    state_query = f"""
        SELECT
            state,
            AVG(TRY_CAST(price_rm AS DOUBLE))    AS mean_price,
            MEDIAN(TRY_CAST(price_rm AS DOUBLE)) AS median_price,
            COUNT(*)                              AS count
        FROM read_csv_auto('{CSV_FILE}', ignore_errors=true, null_padding=true)
        WHERE REGEXP_MATCHES(property_type, 'house|bungalow|terrace|semi.d|villa|townhouse', 'i')
          AND price_rm IS NOT NULL
          AND state    IS NOT NULL
        GROUP BY state
        ORDER BY mean_price DESC
    """

    # Price bucket distribution
    bucket_query = f"""
        SELECT
            CASE
                WHEN TRY_CAST(price_rm AS DOUBLE) <  200000  THEN '< RM 200k'
                WHEN TRY_CAST(price_rm AS DOUBLE) <  500000  THEN 'RM 200k-500k'
                WHEN TRY_CAST(price_rm AS DOUBLE) < 1000000  THEN 'RM 500k-1M'
                ELSE '> RM 1M'
            END AS price_bucket,
            COUNT(*) AS count
        FROM read_csv_auto('{CSV_FILE}', ignore_errors=true, null_padding=true)
        WHERE REGEXP_MATCHES(property_type, 'house|bungalow|terrace|semi.d|villa|townhouse', 'i')
          AND price_rm IS NOT NULL
        GROUP BY price_bucket
        ORDER BY price_bucket
    """

    # Total house count
    count_query = f"""
        SELECT COUNT(*) AS n
        FROM read_csv_auto('{CSV_FILE}', ignore_errors=true, null_padding=true)
        WHERE REGEXP_MATCHES(property_type, 'house|bungalow|terrace|semi.d|villa|townhouse', 'i')
    """

    df_state   = con.execute(state_query).df()
    df_buckets = con.execute(bucket_query).df()
    n_houses   = con.execute(count_query).fetchone()[0]
    con.close()

    return {'df_state': df_state, 'df_buckets': df_buckets, 'n_houses': n_houses}


metrics_duckdb, result_duckdb = run_benchmark(
    'duckdb',
    duckdb_pipeline,
    n_rows_fn=lambda r: r['n_houses'],
)
all_metrics.append(metrics_duckdb)

# ── Print results ─────────────────────────────────────────────────────────────
df_state_d   = result_duckdb['df_state']
df_buckets_d = result_duckdb['df_buckets']
n_houses_d   = result_duckdb['n_houses']

print('=' * 50)
print('  DUCKDB OPTIMIZATION RESULTS')
print('=' * 50)
print(f'  Total Processing Time : {metrics_duckdb["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_duckdb["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_duckdb["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_duckdb["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {n_houses_d:,}')
print('=' * 50)

print(f'\nTotal Houses Filtered: {n_houses_d:,} rows')

print('\nPrice Summary by State for Houses:')
print(f'{"State":<20} {"Mean Price":>18} {"Median Price":>18} {"Count":>8}')
for _, row in df_state_d.head(5).iterrows():
    print(f'{str(row["state"]):<20} RM {row["mean_price"]:>15,.2f} RM {row["median_price"]:>15,.2f} {int(row["count"]):>8,}')

print('\nDistribution by Price Bucket:')
bucket_order = ['< RM 200k', 'RM 200k-500k', 'RM 500k-1M', '> RM 1M']
bucket_map   = dict(zip(df_buckets_d['price_bucket'], df_buckets_d['count']))
for b in bucket_order:
    print(f'  ◆ {b:<15} : {bucket_map.get(b, 0):>6,} properties')


  Running: duckdb (3 runs)...
    Run 1/3 → 1.3341s | CPU: 107.2% | Mem: 0.15MB | Throughput: 40,965 rec/s | Rows: 54,653
    Run 2/3 → 1.0186s | CPU: 136.4% | Mem: 0.15MB | Throughput: 53,658 rec/s | Rows: 54,653
    Run 3/3 → 1.2869s | CPU: 110.3% | Mem: 0.15MB | Throughput: 42,470 rec/s | Rows: 54,653
    ── Avg: 1.2132s | Min: 1.0186s | Max: 1.3341s
  DUCKDB OPTIMIZATION RESULTS
  Total Processing Time : 1.2132 seconds
  CPU Usage             : 118.0 %
  Memory Usage          : 0.15 MB
  Throughput            : 45,698 records/second
  Rows Processed        : 54,653

Total Houses Filtered: 54,653 rows

Price Summary by State for Houses:
State                        Mean Price       Median Price    Count
Penang               RM    2,017,240.99 RM      890,000.00    3,228
Sabah                RM    1,728,408.90 RM      900,000.00    2,761
Putrajaya            RM    1,482,884.64 RM    1,290,000.00      685
Kuala Lumpur         RM    1,392,537.10 RM      780,000.00    3,123
Selangor   

## Step 8 — Performance Comparison Table

In [8]:
df_perf = pd.DataFrame(all_metrics)
baseline_time = df_perf.loc[df_perf['method'] == 'pandas_baseline', 'time_sec'].values[0]
df_perf['speedup_vs_baseline'] = (baseline_time / df_perf['time_sec']).round(2)

df_runs = pd.DataFrame(all_run_records)

avg_cols = df_perf[['method','time_sec','cpu_percent','memory_mb',
                     'throughput_records_per_sec','speedup_vs_baseline']].rename(columns={
    'time_sec':                   'avg_time_sec',
    'cpu_percent':                'avg_cpu_%',
    'memory_mb':                  'avg_memory_mb',
    'throughput_records_per_sec': 'avg_throughput',
})

df_long = df_runs.merge(avg_cols, on='method', how='left')
df_long = df_long.rename(columns={
    'time_sec':                   'time_sec',
    'cpu_percent':                'cpu_%',
    'memory_mb':                  'memory_mb',
    'throughput_records_per_sec': 'throughput',
})

method_order = ['pandas_baseline', 'pandas_optimized', 'polars', 'duckdb']
df_long['method'] = pd.Categorical(df_long['method'], categories=method_order, ordered=True)
df_long = df_long.sort_values(['method', 'run']).reset_index(drop=True)

df_display = df_long[['method','run',
                       'time_sec','cpu_%','memory_mb','throughput',
                       'avg_time_sec','avg_cpu_%','avg_memory_mb',
                       'avg_throughput','speedup_vs_baseline']].copy()

avg_metric_cols = ['avg_time_sec','avg_cpu_%','avg_memory_mb',
                   'avg_throughput','speedup_vs_baseline']
df_clean = df_display.copy()
for col in avg_metric_cols:
    df_clean[col] = [
        v if (i % 3 == 1) else None
        for i, v in enumerate(df_display[col])
    ]

df_clean.to_csv('performance_after.csv', index=False)
print('\n✅ Saved: performance_after.csv')

df_clean_indexed = df_clean.set_index(['method','run'])

def row_color(row_idx):
    base   = ['#fff9e6','#ffffff','#f0f9ff','#f5fff5']
    shade  = ['#ffe9b0','#ffffcc','#d6eeff','#d6ffd6']
    group  = row_idx // 3
    pos    = row_idx % 3
    color  = shade[group % 4] if pos == 1 else base[group % 4]
    return f'background-color: {color}'

print(f"\n{'='*90}")
print(f"  FULL BENCHMARK RESULTS — {total_records:,} records | 3 runs each")
print(f"  Avg columns shown on Run 2 (middle row) per method")
print(f"{'='*90}\n")

display(df_clean_indexed.style
    .format({
        'time_sec':            '{:.4f}s',
        'cpu_%':               '{:.1f}%',
        'memory_mb':           '{:.2f}',
        'throughput':          '{:,.0f}',
        'avg_time_sec':        '{:.4f}s',
        'avg_cpu_%':           '{:.1f}%',
        'avg_memory_mb':       '{:.2f}',
        'avg_throughput':      '{:,.0f}',
        'speedup_vs_baseline': '{:.2f}x',
    }, na_rep='')
    .apply(lambda col: [row_color(i) for i in range(len(col))], axis=0)
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-weight','bold'),('background-color','#e8e8e8'),
                  ('text-align','center')]
    }, {
        'selector': 'tr:nth-child(3n)',
        'props': [('border-bottom','2px solid #aaa')]
    }])
    .highlight_min(subset=['avg_time_sec'],   color='#b6ffb6')
    .highlight_max(subset=['avg_time_sec'],   color='#ffb6b6')
    .highlight_max(subset=['avg_throughput','speedup_vs_baseline'], color='#b6ffb6')
)

fastest = df_perf.loc[df_perf['time_sec'].idxmin()]
print(f"\n{'='*90}")
print(f"  🏆 Fastest : {fastest['method']}  ({fastest['time_sec']}s avg)")
print(f"     Speedup  : {fastest['speedup_vs_baseline']}x faster than pandas baseline")
print(f"     Throughput: {fastest['throughput_records_per_sec']:,.0f} records/sec")
print(f"{'='*90}")

print(f'\n{"-"*60}\n⬇️  Triggering local computer browser downloads...\n{"-"*60}')
for file_name in ['performance_before.csv', 'performance_after.csv']:
    if os.path.exists(file_name):
        print(f"⬇️ Downloading {file_name} to your computer...")
        files.download(file_name)
    else:
        print(f"⚠️ Warning: {file_name} not found in workspace environment.")


✅ Saved: performance_after.csv

  FULL BENCHMARK RESULTS — 100,442 records | 3 runs each
  Avg columns shown on Run 2 (middle row) per method




  🏆 Fastest : polars  (0.8368s avg)
     Speedup  : 7.97x faster than pandas baseline
     Throughput: 65,861 records/sec

------------------------------------------------------------
⬇️  Triggering local computer browser downloads...
------------------------------------------------------------
⬇️ Downloading performance_before.csv to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Downloading performance_after.csv to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>